# ReSnap v5b: evaluasi final (sekali jalan)

Model v5 final: **v5b** (dipilih dengan pengecualian tercatat: gagal syarat 4 karena AP50 holdout Open Images jamur turun 6.5 poin
dan kentang turun 20 poin pada 13 objek dibanding v5a; lebih baik dari v5a dan v4b di val, deteksi palsu foto rempah, dan gap).
Pilihan ini dibuat **sebelum** notebook ini dijalankan. Angka di sini tidak dipakai untuk memilih model.

Yang dievaluasi:
1. **Test lama (654 foto)**: v4b dan v5b, supaya bisa dibandingkan langsung dengan angka laporan v4b.
2. **Test eksternal rimpang (314 foto rival)**: v4b, v5a, v5b, termasuk deteksi palsu di 202 foto rempah lain.
3. Export `best_v5b.onnx`.

Kaggle: GPU T4, Internet on. Input: output notebook **ReSnap v5b - Rempah Relz**, output notebook **ReSnap v5a - Data Train**,
dan dataset **resnap-v4**. Waktu sekitar 15 menit.

In [ ]:
%pip -q install ultralytics==8.3.228 onnx onnxslim

In [ ]:
import json, math, random, shutil, subprocess, sys, zipfile
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageOps

assert torch.cuda.is_available(), 'GPU tidak aktif. Settings > Accelerator > GPU T4.'
DEVICE = '0'
INPUT, OUTPUT, TMP = Path('/kaggle/input'), Path('/kaggle/working'), Path('/tmp/resnap')
OUT = OUTPUT / 'evaluasi_final_v5b'
for p in (OUT, TMP):
    p.mkdir(parents=True, exist_ok=True)
REPO = TMP / 'resnap-cv'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/AffanAbya30/resnap-cv.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'scripts'))
import data_v4 as dv
from evaluate_export import evaluate
from ultralytics import YOLO

V4B = REPO / 'models/best.pt'
V5A = next(INPUT.rglob('best_v5a.pt'), None)
V5B = next(INPUT.rglob('best_v5b.pt'), None)
assert V5A and V5B, 'Tambahkan Input output notebook v5a dan v5b'
print('sha256 v5b:', dv.sha256_file(V5B))

def first_dir(pred):
    hits = sorted((p for p in INPUT.rglob('*') if p.is_dir() and pred(p)), key=lambda p: len(p.parts))
    return hits[0] if hits else None
BASE = first_dir(lambda p: (p / 'test/images').is_dir() and (p / 'train/images').is_dir() and not (p / 'holdout_rimpang').exists())
assert BASE, 'Input resnap-v4 belum ditambahkan'
EXT = next((p for p in INPUT.rglob('test_eksternal_rimpang') if (p / 'images').is_dir()), None)
if EXT is None:
    z = next(INPUT.rglob('dataset_v5a.zip'), None)
    assert z, 'dataset_v5a.zip tidak ditemukan'
    with zipfile.ZipFile(z) as f:
        f.extractall(TMP / 'd5', [m for m in f.namelist() if m.startswith('test_eksternal_rimpang/')])
    EXT = TMP / 'd5' / 'test_eksternal_rimpang'
print('test lama:', len(list((BASE / 'test/images').iterdir())), 'foto | eksternal:', len(list((EXT / 'images').iterdir())), 'foto')
NAMES = dict(enumerate(dv.CLASS_NAMES))
num = lambda d: {k: (round(float(v), 4) if isinstance(v, (int, float)) else v) for k, v in d.items()}

## 1. Test lama (654 foto): v4b vs v5b

In [ ]:
Y_TEST = dv.write_yaml(TMP / 'test.yaml', BASE, 'test/images', 'test/images')
T = {n: evaluate(str(w), Y_TEST, 'val', OUT, f'{n}_test', DEVICE) for n, w in (('v4b', V4B), ('v5b', V5B))}
tab = pd.DataFrame(T).loc[['images', 'precision', 'recall', 'f1', 'mAP50', 'mAP75', 'mAP50_95']]
tab['selisih_poin'] = 100 * (tab.v5b - tab.v4b); tab.loc['images', 'selisih_poin'] = 0
display(tab.style.format(precision=3).set_caption('Test lama 654 foto'))
pc = {n: pd.read_csv(OUT / f'{n}_test' / 'per_class.csv').set_index('class_name') for n in T}
per = pd.DataFrame({'objek': pc['v5b']['instances'], 'AP50_v4b': pc['v4b']['AP50'], 'AP50_v5b': pc['v5b']['AP50']})
per['selisih_poin'] = 100 * (per.AP50_v5b - per.AP50_v4b)
display(per.sort_values('selisih_poin').style.format(precision=3).set_caption('AP50 per kelas, test lama'))
per.to_csv(OUT / 'test_lama_per_kelas.csv'); tab.to_csv(OUT / 'test_lama_keseluruhan.csv')

## 2. Test eksternal rimpang (rival): v4b, v5a, v5b

In [ ]:
Y_EXT = dv.write_yaml(TMP / 'ext.yaml', EXT, 'images', 'images')
MODELS = {'v4b': V4B, 'v5a': V5A, 'v5b': V5B}
X = {n: evaluate(str(w), Y_EXT, 'val', OUT, f'{n}_eksternal', DEVICE) for n, w in MODELS.items()}
xt = pd.DataFrame(X).loc[['precision', 'recall', 'f1', 'mAP50', 'mAP50_95']]
display(xt.style.format(precision=3).set_caption('Test eksternal rimpang'))
xpc = {n: pd.read_csv(OUT / f'{n}_eksternal' / 'per_class.csv').set_index('class_name') for n in MODELS}
xper = pd.DataFrame({n: xpc[n].reindex(['ginger', 'turmeric', 'galangal'])['AP50'] for n in MODELS})
display(xper.style.format(precision=3).set_caption('AP50 rimpang, test eksternal'))

imgs = sorted(p for p in (EXT / 'images').iterdir() if p.suffix.lower() in dv.IMG_EXT)
neg = [p for p in imgs if not (EXT / 'labels' / f'{p.stem}.txt').exists()
       or not (EXT / 'labels' / f'{p.stem}.txt').read_text().strip()]
FP = {}
for n, w in MODELS.items():
    m = YOLO(str(w)); hit, tot, cls = 0, 0, Counter()
    for i in range(0, len(neg), 16):
        for r in m.predict([str(p) for p in neg[i:i + 16]], conf=0.5, iou=0.7, imgsz=640, device=DEVICE, verbose=False, stream=True):
            k = len(r.boxes); tot += k; hit += k > 0; cls.update(NAMES[int(c)] for c in r.boxes.cls.cpu().numpy())
    FP[n] = {'foto_rempah_lain': len(neg), 'foto_dengan_deteksi': int(hit), 'persen': round(100 * hit / len(neg), 1),
             'deteksi_total': int(tot), 'kelas': dict(cls.most_common(5))}
    del m; torch.cuda.empty_cache()
display(pd.DataFrame(FP).T.style.set_caption('Deteksi palsu (conf >= 0.5) di foto rempah lain'))

## 3. Export ONNX dan simpan hasil

In [ ]:
# Input Kaggle read-only: salin dulu ke /kaggle/working, export dari salinan
pt_copy = OUTPUT / 'best_v5b.pt'
shutil.copy2(V5B, pt_copy)
onnx_path = YOLO(str(pt_copy)).export(format='onnx', imgsz=640, opset=12, simplify=True)
print('ONNX:', onnx_path)
res = {'model': 'v5b', 'sha256_pt': dv.sha256_file(V5B),
       'catatan_pemilihan': 'Dipilih sebelum evaluasi ini, dengan pengecualian tercatat pada syarat 4 (holdout Open Images).',
       'test_lama': {n: num(v) for n, v in T.items()}, 'test_eksternal': {n: num(v) for n, v in X.items()},
       'ap50_rimpang_eksternal': xper.round(4).to_dict(orient='index'), 'deteksi_palsu_rempah_lain': FP}
(OUT / 'ringkasan_final.json').write_text(json.dumps(res, indent=2, ensure_ascii=False))
zp = OUTPUT / 'evaluasi_final_v5b.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file() and f.suffix in ('.json', '.csv', '.png') and 'batch' not in f.name:
            z.write(f, f.relative_to(OUT))
print(json.dumps({k: v for k, v in res.items() if k != 'test_eksternal'}, indent=1, ensure_ascii=False)[:3000])
print(zp, round(zp.stat().st_size / 1e6, 2), 'MB')